In [1]:
%pip install -q -U     "gradio==6.28.0"     "strands-agents==1.57.0"     "strands-agents-tools"     "pyowm==3.5.0"     "tavily-python==0.8.4"     "pydantic==2.13.5"     "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=False)

REQUIRED_KEYS = (
    "GROQ_API_KEY",
    "TAVILY_API_KEY",
    "OPENWEATHER_API_KEY",
)

missing = [name for name in REQUIRED_KEYS if not os.getenv(name, "").strip()]

if missing:
    raise RuntimeError(
        "Missing required .env key(s): "
        + ", ".join(missing)
        + ". Add them to your .env file and rerun this cell."
    )

GROQ_API_KEY = os.environ["GROQ_API_KEY"].strip()
TAVILY_API_KEY = os.environ["TAVILY_API_KEY"].strip()
OPENWEATHER_API_KEY = os.environ["OPENWEATHER_API_KEY"].strip()

print("GROQ_API_KEY configured")
print("TAVILY_API_KEY configured")
print("OPENWEATHER_API_KEY configured")

GROQ_API_KEY configured
TAVILY_API_KEY configured
OPENWEATHER_API_KEY configured


In [3]:
import base64
import html
import json
import logging
import re
import tempfile
from pathlib import Path
from typing import List
from urllib.parse import quote, urlparse

import gradio as gr
from pydantic import BaseModel, Field
from pyowm import OWM
from tavily import TavilyClient

from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands_tools import calculator

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)

logger = logging.getLogger("travel-planner")

In [4]:
owm = OWM(OPENWEATHER_API_KEY)
weather_manager = owm.weather_manager()

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print("PyOWM WeatherManager initialized")
print("Tavily client initialized")


PyOWM WeatherManager initialized
Tavily client initialized


In [5]:
@tool
def get_weather(city: str) -> dict:
    # Get current weather for a city using PyOWM/OpenWeather.
    if not city or not city.strip():
        return {"error": "City name cannot be empty."}

    city = city.strip()

    try:
        observation = weather_manager.weather_at_place(city)
        location = observation.location
        weather = observation.weather
        temperature = weather.temperature(unit="celsius")

        return {
            "city": location.name,
            "country": location.country,
            "temperature_c": temperature.get("temp"),
            "feels_like_c": temperature.get("feels_like"),
            "humidity_percent": weather.humidity,
            "description": weather.detailed_status,
            "wind_speed_mps": weather.wind().get("speed", 0),
        }

    except Exception as exc:
        logger.error("Weather lookup failed: %s", exc)
        return {"error": f"Could not retrieve weather for '{city}'."}


@tool
def search_attractions(query: str) -> dict:
    # Search Tavily for attractions and entrance-fee information.
    if not query or not query.strip():
        return {
            "error": "Search query cannot be empty.",
            "results": [],
        }

    try:
        response = tavily_client.search(
            query=query.strip(),
            search_depth="basic",
            max_results=6,
        )

        results = [
            {
                "title": result.get("title", ""),
                "content": result.get("content", ""),
                "url": result.get("url", ""),
            }
            for result in response.get("results", [])
        ]

        return {
            "query": query.strip(),
            "results": results,
        }

    except Exception as exc:
        logger.error("Tavily search failed: %s", exc)
        return {
            "error": "Web search failed.",
            "results": [],
        }


In [6]:
class WeatherInfo(BaseModel):
    temperature_c: float
    feels_like_c: float
    humidity_percent: int
    description: str


class Attraction(BaseModel):
    name: str
    activity: str
    estimated_cost: float = Field(ge=0)
    currency: str
    source_url: str


class TravelPlan(BaseModel):
    city: str
    country: str
    weather: WeatherInfo

    morning: Attraction
    afternoon: Attraction
    evening: Attraction

    total_visit_cost: float = Field(ge=0)
    currency: str

    travel_tips: List[str]


In [7]:
GROQ_MODEL = os.getenv(
    "GROQ_MODEL",
    "qwen/qwen3.8-27b",
).strip() or "qwen/qwen3.8-27b"

if GROQ_MODEL in {
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile",
}:
    GROQ_MODEL = "qwen/qwen3.8-27b"

model = OpenAIModel(
    model_id=GROQ_MODEL,
    client_args={
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
    },
    params={
        "temperature": 0.2,
        "max_tokens": 2500,
        "reasoning_effort": "none",
    },
)

SYSTEM_PROMPT = '''
You are a travel planning assistant.

Create a practical one-day itinerary for the city requested by the user.

You have THREE required tools:

1. get_weather
   - Gets current weather for the destination.

2. search_attractions
   - Searches the web for attractions, preferences, and entrance fees.

3. calculator
   - Performs the final arithmetic.

MANDATORY WORKFLOW
==================

1. Identify the destination city.
2. Call get_weather once for that destination.
3. Call search_attractions EXACTLY ONCE with one broad query covering the city,
   the user's preferences, three attractions, and entrance-fee information.
4. Use only the returned search evidence; do not search again.
5. Select EXACTLY THREE popular attractions.
6. Assign one attraction to Morning, Afternoon, and Evening.
7. Determine an estimated entrance/visit cost for each attraction from the search evidence.
8. Use the SAME currency for all three attraction costs.
9. Call calculator once to add the three costs.
10. total_visit_cost MUST equal the calculator result.
11. Return the final result using the TravelPlan schema.

RULES
=====

- Do not invent current weather.
- Weather must come from get_weather.
- Attraction information and source URLs must come from Tavily results.
- Select exactly three distinct attractions.
- Use calculator for the final cost calculation.
- Do not mix currencies.
- If prices vary, use a reasonable adult visitor estimate and mention that uncertainty in tips.
- Keep the route realistic for one day.
- Provide practical travel tips.
- Never fabricate source URLs.
'''

travel_agent = Agent(
    model=model,
    tools=[
        get_weather,
        search_attractions,
        calculator,
    ],
    system_prompt=SYSTEM_PROMPT,
)

print(f"Groq model initialized: {GROQ_MODEL}")
print("Strands agent initialized with weather, search, and calculator tools")

Groq model initialized: qwen/qwen3.8-27b
Strands agent initialized with weather, search, and calculator tools


In [8]:
def _conversation_context(history: list | None) -> str:
    if not history:
        return ""

    clean = []

    for item in history[-6:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = str(item.get("content", ""))[:1200]

            if role in {"user", "assistant"} and content:
                clean.append({
                    "role": role,
                    "content": content,
                })

    if not clean:
        return ""

    return (
        "\nRecent conversation context "
        "(use only for destination/preferences):\n"
        + json.dumps(clean, ensure_ascii=False)
    )


def generate_trip(message: str, history: list | None = None) -> TravelPlan:
    if not message or not message.strip():
        raise ValueError("Please provide a destination city.")

    prompt = f'''
Create a one-day travel itinerary for this request:

{message.strip()}

{_conversation_context(history)}

Use all three required tools before producing the final TravelPlan.
'''

    try:
        response = travel_agent(
            prompt,
            structured_output_model=TravelPlan,
        )
    except Exception as exc:
        error_text = str(exc).casefold()
        if (
            "429" in error_text
            or "resource_exhausted" in error_text
            or "quota" in error_text
            or "too many requests" in error_text
        ):
            raise RuntimeError(
                "Groq API quota or rate limit reached. Wait briefly, check your "
                "Groq plan, or set GROQ_MODEL to an available model in .env."
            ) from exc
        raise

    plan = response.structured_output

    if not isinstance(plan, TravelPlan):
        plan = TravelPlan.model_validate(plan)

    attractions = [
        plan.morning,
        plan.afternoon,
        plan.evening,
    ]

    if len({item.name.strip().casefold() for item in attractions}) != 3:
        raise ValueError(
            "The agent returned duplicate attractions. Please try again."
        )

    if any(
        item.currency.upper() != plan.currency.upper()
        for item in attractions
    ):
        raise ValueError(
            "The agent mixed currencies. Please try again."
        )

    calculated = round(
        sum(item.estimated_cost for item in attractions),
        2,
    )

    if round(plan.total_visit_cost, 2) != calculated:
        logger.warning(
            "Structured total %.2f did not match attraction sum %.2f; "
            "using deterministic sum for display.",
            plan.total_visit_cost,
            calculated,
        )
        plan.total_visit_cost = calculated

    return plan

In [9]:
CSS = """
:root {--paper:#f6f3ec;--ink:#193c32;--muted:#69766c;--line:#dedfd4;--accent:#dc794b}
body,.gradio-container{background:var(--paper)!important;color:var(--ink)!important;font-family:'Segoe UI',Arial,sans-serif!important}
.gradio-container{max-width:1260px!important;margin:auto!important;padding:0 36px 24px!important}
.dark{--background-fill-primary:var(--paper)!important;--body-text-color:var(--ink)!important}
footer{display:none!important}
#topbar{margin:0!important;padding:0!important;border:0!important;background:transparent!important}
.masthead{display:flex;align-items:center;justify-content:space-between;height:86px;border-bottom:1px solid var(--line)}
.brand{font:700 34px Georgia,serif;letter-spacing:-2px;display:flex;align-items:center;gap:11px}
.brand-mark{display:grid;place-items:center;width:30px;height:30px;border:1px solid #8d9b88;border-radius:50%;font:24px Georgia;letter-spacing:0}
.brand i{color:var(--accent);font-style:normal}
.mast-note{font-size:11px;letter-spacing:2.3px;text-transform:uppercase;color:var(--muted)}
.mast-edition{font-size:12px;color:var(--muted)}
#hero{border:0!important;background:transparent!important;padding:0!important;margin:14px 0 20px!important}
.hero{display:grid;grid-template-columns:1.05fr 1fr;gap:55px;align-items:center;padding:20px 0}
.eyebrow{font:600 10px 'Segoe UI',sans-serif;letter-spacing:2px;text-transform:uppercase;color:var(--muted)}
.eyebrow .dot{display:inline-block;width:6px;height:6px;background:var(--accent);border-radius:50%;margin-right:8px}
.hero h1{font:400 clamp(43px,4.6vw,66px)/1.02 Georgia,serif;letter-spacing:-2.9px;margin:17px 0 17px;color:var(--ink)}
.hero h1 em{font-weight:400;color:#b8613e}
.hero p{max-width:405px;font-size:14px;line-height:1.7;color:var(--muted);margin:0}
.hero-image{position:relative;overflow:hidden;border-radius:5px 75px 5px 5px;height:258px}
.hero-image img{width:100%;height:100%;object-fit:cover;object-position:center 53%}
.image-caption{position:absolute;bottom:17px;left:20px;color:#fff9e7;font:italic 19px Georgia;text-shadow:0 1px 5px #183c32}
.stamp{position:absolute;right:15px;bottom:13px;color:#fff9e7;border:1px solid #fff9e780;border-radius:50%;width:55px;height:55px;text-align:center;padding-top:9px;font-size:8px;letter-spacing:1px;transform:rotate(12deg)}
.stamp b{display:block;font:19px Georgia}
#workspace{gap:24px!important;align-items:stretch}
#conversation-panel{border:1px solid var(--line)!important;border-radius:12px!important;background:#fffefa!important;padding:20px!important;gap:12px!important;min-width:300px!important}
.panel-heading{display:flex;align-items:center;justify-content:space-between}
.panel-heading h2{font:400 24px Georgia;letter-spacing:-.4px;margin:0}
.small-pill{font-size:9px;text-transform:uppercase;letter-spacing:1.1px;border:1px solid var(--line);padding:5px 7px;border-radius:5px;color:var(--muted)}
.subhead{font-size:12px;color:var(--muted);line-height:1.65;margin:6px 0 0}
#chat{border:0!important;background:transparent!important;box-shadow:none!important;padding:0!important}
#chat,#quick-cities{display:none!important}
#chat .bubble{border-radius:10px!important;box-shadow:none!important;max-width:100%!important;font-size:13px!important;line-height:1.6!important}
#chat .user{background:#e7eddf!important;border:0!important;color:var(--ink)!important}
#chat .bot{background:#f5f4ed!important;border:0!important;color:var(--ink)!important}
#chat .message-row{padding:6px 0!important}
#chat .prose{font-size:13px!important;color:var(--ink)!important}
#chat .prose h2{font:22px Georgia!important;margin-top:10px!important}
#chat .prose h3{font:18px Georgia!important}
#chat .prose a{color:#386747!important}
#quick-cities{gap:7px!important}
.city-chip{min-width:0!important;border:1px solid var(--line)!important;background:#fffefa!important;border-radius:30px!important;padding:8px 10px!important;color:var(--ink)!important;font-size:11px!important;box-shadow:none!important}
.city-chip:hover{background:#eaf0e3!important;border-color:#9da992!important}
#composer{background:transparent!important;border:0!important;padding:0!important}
#composer textarea{background:#fffefa!important;border:1px solid #c9cfbd!important;border-radius:8px!important;padding:13px!important;font-size:13px!important;color:var(--ink)!important;min-height:73px!important;box-shadow:none!important}
#composer textarea:focus{outline:2px solid #a0b395!important;outline-offset:2px}
#send{background:var(--ink)!important;color:#fffdf4!important;border:0!important;border-radius:7px!important;font-size:13px!important;min-height:44px!important;box-shadow:none!important}
#send:hover{background:#285542!important}
#send:disabled{opacity:.6!important}
#chat-actions{gap:10px!important}
.quiet-button{background:transparent!important;border:0!important;font-size:11px!important;color:var(--muted)!important;min-width:0!important;padding:5px!important;box-shadow:none!important}
.quiet-button:hover{text-decoration:underline!important;color:var(--ink)!important}
.status{font-size:11px;line-height:1.5;color:var(--muted);min-height:17px;display:flex;gap:7px;align-items:center}
.status .status-dot{width:5px;height:5px;display:inline-block;background:#658968;border-radius:50%;flex-shrink:0}
.status.working .status-dot{background:var(--accent);animation:breathe 1s infinite alternate}
.status.error{color:#a04928}
@keyframes breathe{to{opacity:.25}}
#itinerary-panel{padding:0!important;gap:0!important;background:transparent!important;min-width:320px!important}
#itinerary{padding:0!important;border:0!important;background:transparent!important}
.trip-board{background:#fffefa;border:1px solid var(--line);border-radius:12px;overflow:hidden;min-height:598px}
.board-top{display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--line);padding:18px 24px}
.board-top .eyebrow{color:#59745b}
.board-top span:last-child{font-size:10px;color:var(--muted)}
.empty-body{padding:35px 34px 24px;text-align:center}
.compass{position:relative;width:71px;height:71px;border:1px solid #bdc8b4;border-radius:50%;margin:0 auto 18px;display:grid;place-items:center;color:#8b9b81;background:#f0f3e9}
.compass svg{width:35px;height:35px}
.empty-body h2{font:400 32px/1.12 Georgia;letter-spacing:-.9px;margin:0 0 13px}
.empty-body p{font-size:13px;color:var(--muted);line-height:1.7;max-width:355px;margin:0 auto}
.empty-route{display:flex;align-items:center;justify-content:center;gap:0;padding:29px 6px 17px}
.empty-stop{width:110px;color:#7f8e77;font-size:10px;letter-spacing:1px;text-transform:uppercase}
.empty-stop b{display:block;font:22px Georgia;letter-spacing:0;background:#f0f1e8;border:1px solid #d9dfce;border-radius:50%;width:37px;height:37px;line-height:35px;margin:0 auto 10px;color:#8a9b80}
.route-dash{border-top:1px dashed #bac6af;width:38px;margin-top:-22px}
.promise-row{margin:14px 24px 24px;display:grid;grid-template-columns:repeat(3,1fr);border-top:1px solid var(--line);padding-top:22px;gap:12px}
.promise-row div{font-size:11px;color:var(--muted);line-height:1.65}
.promise-row b{display:block;color:var(--ink);font-size:12px;font-weight:600;margin-bottom:4px}
.trip-intro{padding:24px 24px 19px}
.trip-intro h2{font:400 36px/1.1 Georgia;letter-spacing:-1.2px;margin:8px 0 7px}
.trip-intro p{font-size:12px;line-height:1.7;color:var(--muted);margin:0}
.sample-banner{padding:9px 24px;background:#fbefdb;border-bottom:1px solid #eadcc1;font-size:11px;color:#896533;line-height:1.6}
.weather-strip{display:flex;align-items:center;gap:16px;background:#edf1e5;border:1px solid #dce3cf;margin:0 24px 23px;border-radius:8px;padding:13px 16px}
.weather-symbol{font:30px Georgia;color:#a97c44}
.weather-temp{font:26px Georgia;white-space:nowrap;color:var(--ink)}
.weather-copy{font-size:11px;color:#61725e;line-height:1.6}
.weather-copy b{display:block;color:var(--ink);font-size:12px;font-weight:500}
.weather-side{font-size:10px;text-align:right;color:#657560;margin-left:auto;line-height:1.8}
.itinerary-label{padding:0 24px 10px;display:flex;justify-content:space-between;color:var(--muted);font-size:10px;letter-spacing:1.6px;text-transform:uppercase}
.stops{padding:0 24px}
.stop{display:grid;grid-template-columns:31px 1fr;gap:13px;padding:14px 0 16px;border-top:1px solid var(--line)}
.stop-no{width:29px;height:29px;border:1px solid #cbd4bf;background:#eef2e6;border-radius:50%;display:grid;place-items:center;font:13px Georgia;color:#617455;margin-top:1px}
.stop-time{font-size:9px;letter-spacing:1.6px;text-transform:uppercase;color:#a76b44;line-height:1.5;margin-bottom:4px}
.stop h3{font:400 23px/1.2 Georgia;letter-spacing:-.5px;margin:0 0 7px;color:var(--ink)}
.stop p{font-size:12px;line-height:1.65;color:var(--muted);margin:0 0 8px}
.stop-meta{display:flex;align-items:center;gap:10px;flex-wrap:wrap;font-size:10px;color:var(--muted)}
.stop-meta .cost-tag{background:#f1efe5;border-radius:4px;padding:4px 6px;color:#5f684e}
.stop-meta a{color:#426d4a!important;text-decoration:none!important;border-bottom:1px solid #bfd0b4}
.cost-note{font-size:10px!important;color:#8b826b!important;margin-top:6px!important}
.budget{background:#1d3c32;color:#f5f3e6;padding:21px 24px;display:flex;align-items:center;justify-content:space-between;gap:20px}
.budget .budget-label{font-size:9px;text-transform:uppercase;letter-spacing:1.5px;color:#bfceba}
.budget small{display:block;font-size:10px;color:#bdcbbb;line-height:1.6;margin-top:6px;max-width:330px}
.budget strong{font:27px Georgia;white-space:nowrap;font-weight:400}
.field-notes{padding:20px 24px;background:#f6f5ed}
.field-notes h4{font:18px Georgia;margin:0 0 8px}
.field-notes ul{margin:0;padding-left:17px;font-size:11px;line-height:1.8;color:#67715f}
#download{font-size:12px!important;color:var(--ink)!important;border:1px solid var(--line)!important;background:#eaf0e3!important;border-radius:7px!important;margin-top:12px!important;box-shadow:none!important}
.page-foot{display:flex;justify-content:space-between;gap:20px;border-top:1px solid var(--line);margin-top:28px;padding-top:17px;color:#7c8477;font-size:10px;line-height:1.7}
button:focus-visible,a:focus-visible{outline:2px solid #ba7449!important;outline-offset:3px!important}
@media(max-width:800px){.gradio-container{padding:0 18px 20px!important}.masthead{height:70px}.mast-note{display:none}.hero{gap:22px}.hero h1{font-size:44px}.hero-image{height:235px;border-radius:4px 48px 4px 4px}.hero p{font-size:12px}.hero .eyebrow{font-size:8px}.hero h1{letter-spacing:-2px}#workspace{flex-direction:column!important}#conversation-panel,#itinerary-panel{width:100%!important;min-width:0!important;flex-basis:auto!important}.trip-board{min-height:0}.page-foot{font-size:9px}}
@media(max-width:520px){.hero{grid-template-columns:1fr;gap:23px;padding-top:12px}.hero h1{font-size:51px;margin:14px 0}.hero p{max-width:340px;font-size:13px}.hero-image{height:185px;border-radius:4px 45px 4px 4px}.hero-image img{object-position:center 54%}.hero .eyebrow{font-size:9px}.mast-edition{font-size:10px}.brand{font-size:29px}#hero{margin-bottom:12px!important}#conversation-panel{padding:17px!important}.empty-body{padding:30px 20px 10px}.empty-body h2{font-size:30px}.promise-row{gap:9px}.promise-row div{font-size:10px}.promise-row b{font-size:11px}.weather-side{display:none}.trip-intro h2{font-size:32px}.budget{flex-wrap:wrap;gap:10px}.board-top{padding:17px 20px}.page-foot{flex-direction:column;gap:4px}}
@media(prefers-reduced-motion:reduce){*{animation:none!important;scroll-behavior:auto!important}}
"""

In [10]:
CSS += """
.planning-status{display:block!important;position:relative;overflow:hidden;padding:12px 13px!important;border:1px solid #d8dfcf;border-radius:10px;background:#eef2e8;color:var(--ink);min-height:74px}
.planning-status:after{content:"";position:absolute;inset:0 auto 0 -35%;width:38%;background:linear-gradient(90deg,transparent,#ffffff66,transparent);transform:skewX(-18deg);animation:planning-sheen 2.8s ease-in-out infinite}
.planning-orbit{position:relative;float:left;width:38px;height:38px;margin:3px 12px 5px 0;border:1px solid #b8c8ac;border-radius:50%}
.planning-orbit:before,.planning-orbit:after{content:"";position:absolute;border-radius:50%;background:var(--accent)}
.planning-orbit:before{width:7px;height:7px;top:4px;left:15px;animation:planning-orbit 2.2s linear infinite}
.planning-orbit:after{width:4px;height:4px;right:5px;bottom:8px;background:#658968;animation:planning-pulse 1.2s ease-in-out infinite}
.planning-copy{position:relative;z-index:1;display:block;line-height:1.35}
.planning-kicker{display:block;font-size:8px;letter-spacing:1.8px;text-transform:uppercase;color:#71806d;margin-bottom:3px}
.planning-copy strong{display:block;font:16px Georgia,serif;font-weight:400;color:var(--ink)}
.planning-copy small{display:block;margin-top:3px;font-size:10px;color:#69766c}
.planning-steps{position:relative;z-index:1;display:flex;clear:both;gap:5px;padding-top:7px}
.planning-steps span{flex:1;height:3px;border-radius:4px;background:#d1d9c8;font-size:0}
.planning-steps span:nth-child(1){animation:planning-step 4.8s ease-in-out infinite}
.planning-steps span:nth-child(2){animation:planning-step 4.8s .7s ease-in-out infinite}
.planning-steps span:nth-child(3){animation:planning-step 4.8s 1.4s ease-in-out infinite}
@keyframes planning-sheen{0%,25%{left:-35%}65%,100%{left:115%}}
@keyframes planning-orbit{to{transform:rotate(360deg) translateX(12px) rotate(-360deg)}}
@keyframes planning-pulse{50%{transform:scale(1.8);opacity:.45}}
@keyframes planning-step{0%,100%{background:#d1d9c8}35%,70%{background:var(--accent)}}
@media(prefers-reduced-motion:reduce){.planning-status:after,.planning-orbit:before,.planning-orbit:after,.planning-steps span{animation:none!important}}
"""

In [11]:
HERO_DATA = '...'

In [12]:
WELCOME = (
    "Hi, I'm your day-trip companion. Tell me a city, and I'll bring together "
    "the current weather, three worthwhile stops, and an estimated entrance-fee total."
    "\n\nYou can also add a preference, like **a slower pace** or **more art**."
)


def _e(value) -> str:
    return html.escape(str(value), quote=True)


def _md(value) -> str:
    return re.sub(r"([\\`*_{}\[\]()#+.!|>~-])", r"\\\1", html.escape(str(value)))


def _ui_safe_url(value: str) -> str:
    try:
        parsed = urlparse(value)
        return (
            value
            if parsed.scheme in {"https", "http"}
            and parsed.netloc
            and not parsed.username
            else ""
        )
    except ValueError:
        return ""


def _money(value: float, currency: str) -> str:
    return f"{value:,.2f} {currency}"


def _status(text: str, state: str = "") -> str:
    return (
        f'<div class="status {_e(state)}" role="status" aria-live="polite">'
        f'<span class="status-dot"></span>{_e(text)}</div>'
    )


def _artwork() -> str:
    if "HERO_DATA" in globals():
        return globals()["HERO_DATA"]

    asset = Path.cwd() / "assets" / "sojourn-hero.png"

    if asset.exists():
        return (
            "data:image/png;base64,"
            + base64.b64encode(asset.read_bytes()).decode("ascii")
        )

    return ""


def hero_html() -> str:
    artwork = _artwork()

    image = (
        f'<img src="{artwork}" '
        'alt="Illustration of a winding trail through green Himalayan hills toward a pagoda"/>'
        if artwork
        else '<div class="hero-fallback">SOJOURN</div>'
    )

    return f'''<section class="hero">
      <div>
        <div class="eyebrow"><span class="dot"></span>Small journeys. Good stories.</div>
        <h1>One city.<br>A day <em>well spent.</em></h1>
        <p>A little local discovery, a thoughtful route, and room to wander.
        Let's make the most of your next day out.</p>
      </div>
      <div class="hero-image">{image}
        <span class="image-caption">Take the scenic route.</span>
        <span class="stamp">A LITTLE<b>wonder</b>EVERY DAY</span>
      </div>
    </section>'''


def empty_board() -> str:
    return '''<section class="trip-board">
      <div class="board-top">
        <span class="eyebrow">Your day, at a glance</span>
        <span>01 DAY / 03 STOPS</span>
      </div>
      <div class="empty-body">
        <div class="compass">
          <svg viewBox="0 0 40 40" fill="none" aria-hidden="true">
            <path d="M28 10 23 23 10 29l6-14z" stroke="currentColor" stroke-width="1.3"/>
            <path d="m16 15 7 8 5-13z" fill="currentColor"/>
            <circle cx="20" cy="20" r="17" stroke="currentColor" stroke-width=".5"/>
          </svg>
        </div>
        <h2>Somewhere new.<br>Something memorable.</h2>
        <p>Choose a city in the chat. Your own little travel journal will take shape right here.</p>
        <div class="empty-route">
          <div class="empty-stop"><b>1</b>Morning</div>
          <div class="route-dash"></div>
          <div class="empty-stop"><b>2</b>Afternoon</div>
          <div class="route-dash"></div>
          <div class="empty-stop"><b>3</b>Evening</div>
        </div>
      </div>
      <div class="promise-row">
        <div><b>Weather-aware</b>Current conditions to help you pack.</div>
        <div><b>Worth a visit</b>Three stops, with sources to explore.</div>
        <div><b>Costs made clear</b>Estimated attraction fees.</div>
      </div>
    </section>'''


def _summary_for(plan: TravelPlan) -> str:
    return (
        f"A balanced one-day route through {plan.city}, with a morning, "
        f"afternoon, and evening stop selected from current web search results."
    )


def sample_plan() -> TravelPlan:
    return TravelPlan.model_validate({
        "city": "Kathmandu",
        "country": "Nepal",
        "currency": "NPR",
        "weather": {
            "temperature_c": 24,
            "feels_like_c": 25,
            "humidity_percent": 62,
            "description": "Partly cloudy — sample weather",
        },
        "morning": {
            "name": "Kathmandu Durbar Square",
            "activity": "Explore the historic courtyards and temple architecture of the old city.",
            "estimated_cost": 1000,
            "currency": "NPR",
            "source_url": "",
        },
        "afternoon": {
            "name": "Garden of Dreams",
            "activity": "Slow the pace with a relaxed walk through the landscaped garden.",
            "estimated_cost": 400,
            "currency": "NPR",
            "source_url": "",
        },
        "evening": {
            "name": "Boudhanath Stupa",
            "activity": "Finish the day with an evening circuit around the stupa.",
            "estimated_cost": 400,
            "currency": "NPR",
            "source_url": "",
        },
        "total_visit_cost": 1800,
        "travel_tips": [
            "This is a visual sample only; weather and prices are illustrative.",
            "Use the live planner to retrieve current weather and web sources.",
            "Meals, transport, guides, and accommodation are not included in the attraction total.",
        ],
    })


def format_plan(plan: TravelPlan, sample: bool = False) -> str:
    prefix = (
        "**SAMPLE PREVIEW — weather and prices are illustrative.**\n\n"
        if sample
        else ""
    )

    result = (
        prefix
        + f"## A day in {_md(plan.city)}, {_md(plan.country)}\n\n"
        + f"{_md(_summary_for(plan))}\n\n"
        + f"**Weather:** {plan.weather.temperature_c:.1f} °C · "
        + f"{_md(plan.weather.description)} · "
        + f"Feels like {plan.weather.feels_like_c:.1f} °C · "
        + f"Humidity {plan.weather.humidity_percent}%\n\n"
    )

    for when, attraction in [
        ("Morning", plan.morning),
        ("Afternoon", plan.afternoon),
        ("Evening", plan.evening),
    ]:
        result += (
            f"### {when} · {_md(attraction.name)}\n"
            f"{_md(attraction.activity)}\n\n"
            f"**Estimated visit cost:** "
            f"{_md(_money(attraction.estimated_cost, attraction.currency))}\n\n"
        )

        url = _ui_safe_url(attraction.source_url)

        if url:
            result += (
                f"[Explore the source]"
                f"({quote(url, safe=':/?=&%#')})\n\n"
            )

    result += (
        f"**Total attraction cost: "
        f"{_md(_money(plan.total_visit_cost, plan.currency))}**\n\n"
        "Meals, transport, guides, and accommodation are excluded. "
        "Actual prices can vary by visitor category.\n\n"
        "**Travel tips**\n"
    )

    result += "\n".join(
        f"- {_md(tip)}"
        for tip in plan.travel_tips
    )

    return result


def render_plan(plan: TravelPlan, sample: bool = False) -> str:
    cards = []

    for idx, (when, attraction) in enumerate(
        [
            ("Morning", plan.morning),
            ("Afternoon", plan.afternoon),
            ("Evening", plan.evening),
        ],
        start=1,
    ):
        url = _ui_safe_url(attraction.source_url)

        source = (
            f'<a href="{_e(url)}" target="_blank" rel="noopener noreferrer">'
            "View source ↗</a>"
            if url
            else "<span>Sample stop</span>"
            if sample
            else "<span>Source unavailable</span>"
        )

        maps = (
            "https://www.google.com/maps/search/?api=1&query="
            + quote(
                f"{attraction.name}, {plan.city}, {plan.country}"
            )
        )

        cards.append(
            f'''<article class="stop">
              <div class="stop-no">{idx}</div>
              <div>
                <div class="stop-time">{when}</div>
                <h3>{_e(attraction.name)}</h3>
                <p>{_e(attraction.activity)}</p>
                <div class="stop-meta">
                  <span class="cost-tag">{_e(_money(attraction.estimated_cost, attraction.currency))}</span>
                  {source}
                  <a href="{_e(maps)}" target="_blank" rel="noopener noreferrer">Find on map ↗</a>
                </div>
              </div>
            </article>'''
        )

    weather = plan.weather

    details = [
        f"Feels like {weather.feels_like_c:.0f}°C",
        f"Humidity {weather.humidity_percent}%",
    ]

    tips = "".join(
        f"<li>{_e(tip)}</li>"
        for tip in plan.travel_tips
    )

    banner = (
        '<div class="sample-banner">'
        "SAMPLE JOURNAL · Illustrative weather & prices. "
        "Plan a trip in the chat for live results.</div>"
        if sample
        else ""
    )

    return f'''<section class="trip-board">
      <div class="board-top">
        <span class="eyebrow">Your day, at a glance</span>
        <span>01 DAY / 03 STOPS</span>
      </div>
      {banner}
      <div class="trip-intro">
        <div class="eyebrow">{_e(plan.country)} · A little local discovery</div>
        <h2>A day in {_e(plan.city)}.</h2>
        <p>{_e(_summary_for(plan))}</p>
      </div>
      <div class="weather-strip">
        <span class="weather-symbol" aria-hidden="true">☼</span>
        <span class="weather-temp">{weather.temperature_c:.0f}°C</span>
        <div class="weather-copy">
          <b>{_e(weather.description)}</b>
          {'Sample observation' if sample else 'OpenWeather · current conditions'}
        </div>
        <div class="weather-side">{'<br>'.join(_e(d) for d in details)}</div>
      </div>
      <div class="itinerary-label">
        <span>The day unfolds</span>
        <span>Room to wander</span>
      </div>
      <div class="stops">{''.join(cards)}</div>
      <div class="budget">
        <div>
          <span class="budget-label">{'Sample' if sample else 'Estimated'} attraction fees</span>
          <small>Attraction/visit costs only. Meals & transport are extra.</small>
        </div>
        <strong>{_e(_money(plan.total_visit_cost, plan.currency))}</strong>
      </div>
      <div class="field-notes">
        <h4>Notes for the road</h4>
        <ul>{tips}</ul>
      </div>
    </section>'''


def export_plan(plan: TravelPlan, sample: bool = False) -> str:
    folder = Path(
        tempfile.mkdtemp(prefix="sojourn-")
    )

    filename = folder / "my-day-with-sojourn.md"

    filename.write_text(
        format_plan(plan, sample),
        encoding="utf-8",
    )

    return str(filename)


In [13]:
_default_status = _status


def _status(text: str, state: str = "") -> str:
    if state != "working":
        return _default_status(text, state)

    return '''<div class="status planning-status" role="status" aria-live="polite">
      <div class="planning-orbit" aria-hidden="true"></div>
      <div class="planning-copy">
        <span class="planning-kicker">BUILDING YOUR DAY</span>
        <strong>Finding the good bits...</strong>
        <small>Reading the weather, places, and price clues.</small>
      </div>
      <div class="planning-steps" aria-hidden="true">
        <span>Weather</span><span>Places</span><span>Fees</span>
      </div>
    </div>'''


In [14]:
def new_trip():
    welcome = [
        {
            "role": "assistant",
            "content": WELCOME,
        }
    ]

    return (
        welcome,
        [],
        empty_board(),
        _status("Ready when you are."),
        gr.update(value=None, visible=False),
        gr.update(value="", interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
    )


def preview_sample():
    plan = sample_plan()

    conversation = [
        {
            "role": "assistant",
            "content": format_plan(
                plan,
                sample=True,
            ),
        }
    ]

    return (
        conversation,
        [],
        render_plan(
            plan,
            sample=True,
        ),
        _status(
            "Sample preview · no API calls made."
        ),
        gr.update(
            value=export_plan(
                plan,
                True,
            ),
            visible=True,
        ),
        gr.update(
            value="",
            interactive=True,
        ),
        gr.update(interactive=True),
        gr.update(interactive=True),
        gr.update(interactive=True),
    )


def submit_trip(message: str, history: list):
    message = (message or "").strip()

    if not message:
        yield (
            gr.skip(),
            gr.skip(),
            gr.skip(),
            _status(
                "Enter a city, such as Kathmandu, Nepal."
            ),
            gr.skip(),
            gr.skip(),
            gr.skip(),
            gr.skip(),
            gr.skip(),
        )
        return

    previous = list(history or [])

    chat = previous + [
        {
            "role": "user",
            "content": message,
        }
    ]

    yield (
        chat,
        previous,
        gr.skip(),
        _status(
            "Planning your day · checking weather, attractions & fees…",
            "working",
        ),
        gr.update(visible=False),
        gr.update(
            value="",
            interactive=False,
        ),
        gr.update(interactive=False),
        gr.update(interactive=False),
        gr.update(interactive=False),
    )

    try:
        plan = generate_trip(
            message,
            previous,
        )

        answer = format_plan(plan)

        complete = chat + [
            {
                "role": "assistant",
                "content": answer,
            }
        ]

        yield (
            complete,
            complete[-8:],
            render_plan(plan),
            _status(
                "Your journal is ready. Ask for a change, or take it with you."
            ),
            gr.update(
                value=export_plan(plan),
                visible=True,
            ),
            gr.update(
                value="",
                interactive=True,
            ),
            gr.update(interactive=True),
            gr.update(interactive=True),
            gr.update(interactive=True),
        )

    except Exception as exc:
        logger.exception(
            "Travel planner error"
        )

        error_text = str(exc).casefold()
        quota_error = (
            "quota" in error_text
            or "429" in error_text
            or "resource_exhausted" in error_text
        )
        safe_message = (
            "Gemini is temporarily out of API quota. Wait for the quota window "
            "to reset or set GEMINI_MODEL in .env to a model with available quota."
            if quota_error
            else (
                "I couldn't complete that itinerary. "
                f"Error: {type(exc).__name__}. "
                "Please try the city again, for example: Kathmandu, Nepal."
            )
        )

        chat.append(
            {
                "role": "assistant",
                "content": safe_message,
            }
        )

        yield (
            chat,
            previous,
            gr.skip(),
            _status(
                "The plan wasn't completed. You can try again.",
                "error",
            ),
            gr.update(visible=False),
            gr.update(
                value="",
                interactive=True,
            ),
            gr.update(interactive=True),
            gr.update(interactive=True),
            gr.update(interactive=True),
        )


def build_app() -> gr.Blocks:
    with gr.Blocks(
        title="Sojourn · A day well spent",
        analytics_enabled=False,
        delete_cache=(3600, 86400),
    ) as demo:

        gr.HTML(
            '<header class="masthead">'
            '<div class="brand">'
            '<span class="brand-mark">✳</span>'
            'sojourn<i>.</i>'
            '</div>'
            '<span class="mast-note">'
            'The art of a good day out'
            '</span>'
            '<span class="mast-edition">'
            'Your personal day-trip journal'
            '</span>'
            '</header>',
            elem_id="topbar",
        )

        gr.HTML(
            hero_html(),
            elem_id="hero",
        )

        history = gr.State([])

        with gr.Row(
            elem_id="workspace"
        ):
            with gr.Column(
                scale=4,
                min_width=300,
                elem_id="conversation-panel",
            ):
                gr.HTML(
                    '<div class="panel-heading">'
                    '<h2>Where shall we go?</h2>'
                    '<span class="small-pill">'
                    'Travel companion'
                    '</span>'
                    '</div>'
                    '<p class="subhead">'
                    'Start with a city. Make the day your own.'
                    '</p>'
                )

                chatbot = gr.Chatbot(
                    value=[
                        {
                            "role": "assistant",
                            "content": WELCOME,
                        }
                    ],
                    height=290,
                    show_label=False,
                    layout="bubble",
                    buttons=["copy"],
                    feedback_options=(),
                    allow_tags=False,
                    sanitize_html=True,
                    elem_id="chat",
                )

                with gr.Row(
                    elem_id="quick-cities"
                ):
                    kathmandu = gr.Button(
                        "Kathmandu ↗",
                        elem_classes="city-chip",
                    )
                    kyoto = gr.Button(
                        "Kyoto ↗",
                        elem_classes="city-chip",
                    )
                    lisbon = gr.Button(
                        "Lisbon ↗",
                        elem_classes="city-chip",
                    )

                destination = gr.Textbox(
                    placeholder=(
                        "A city, a mood, "
                        "a little curiosity…"
                    ),
                    label=(
                        "Your destination "
                        "or follow-up request"
                    ),
                    show_label=False,
                    lines=2,
                    max_lines=4,
                    max_length=500,
                    elem_id="composer",
                )

                send = gr.Button(
                    "Plan my day  →",
                    variant="primary",
                    elem_id="send",
                )

                status = gr.HTML(
                    _status(
                        "Ready when you are."
                    )
                )

                with gr.Row(
                    elem_id="chat-actions"
                ):
                    sample = gr.Button(
                        "Preview a sample",
                        elem_classes="quiet-button",
                    )
                    reset = gr.Button(
                        "New trip ↻",
                        elem_classes="quiet-button",
                    )

            with gr.Column(
                scale=6,
                min_width=320,
                elem_id="itinerary-panel",
            ):
                itinerary = gr.HTML(
                    empty_board(),
                    elem_id="itinerary",
                )

                download = gr.DownloadButton(
                    "Take your itinerary with you  ↓",
                    visible=False,
                    elem_id="download",
                )

        gr.HTML(
            '<div class="page-foot">'
            '<span>'
            'Thoughtfully planned with '
            'Strands Agents · Gemini · '
            'OpenWeather · Tavily'
            '</span>'
            '<span>'
            'Made for the curious. '
            'Built by Sandesh.'
            '</span>'
            '</div>'
        )

        outputs = [
            chatbot,
            history,
            itinerary,
            status,
            download,
            destination,
            send,
            sample,
            reset,
        ]

        options = dict(
            fn=submit_trip,
            inputs=[
                destination,
                history,
            ],
            outputs=outputs,
            concurrency_limit=1,
            concurrency_id="planner",
            show_progress="hidden",
            trigger_mode="once",
        )

        send.click(
            **options,
            api_name="plan_trip",
        )

        destination.submit(
            **options,
            api_name=False,
        )

        sample.click(
            preview_sample,
            outputs=outputs,
            concurrency_id="planner",
            api_name="sample_journal",
            show_progress="hidden",
        )

        reset.click(
            new_trip,
            outputs=outputs,
            concurrency_id="planner",
            api_name="new_trip",
            show_progress="hidden",
        )

        for button, city in [
            (
                kathmandu,
                "Kathmandu, Nepal",
            ),
            (
                kyoto,
                "Kyoto, Japan",
            ),
            (
                lisbon,
                "Lisbon, Portugal",
            ),
        ]:
            button.click(
                lambda city=city: city,
                outputs=destination,
                queue=False,
                api_name=False,
            )

    return demo.queue(
        default_concurrency_limit=1,
        max_size=12,
    )


def launch_app(
    *,
    inline: bool = True,
    inbrowser: bool = False,
    share: bool = False,
    height: int = 1000,
):
    app = build_app()

    theme = gr.themes.Base(
        primary_hue="green",
        neutral_hue="stone",
        font=[
            "Segoe UI",
            "Arial",
            "sans-serif",
        ],
    )

    app.launch(
        css=CSS,
        theme=theme,
        share=share,
        inline=inline,
        inbrowser=inbrowser,
        footer_links=[],
        show_error=False,
        height=height,
    )

    return app

In [ ]:
demo = launch_app(
    inline=True,
    inbrowser=False,
    share=False,
    height=1000,
)


* Running on local URL:  http://127.0.0.1:7860


INFO | HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO | HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


INFO | Creating Strands MetricsClient
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Tool #1: get_weather

Tool #2: search_attractions


...


Tool #3: get_weather


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO | Retrying request to /chat/completions in 42.000000 seconds
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
WARNING | DEPRECATION WARNING: calculator is deprecated. This warning becomes an error log in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.



Tool #4: calculator


INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO | Retrying request to /chat/completions in 43.000000 seconds
INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Tool #5: TravelPlan
